<p class="lead">A model that only its author can run has a readership of one. This unit takes the ROCK1 predictor built earlier in the course and turns it into something a medicinal chemist in another laboratory can use from a browser: a serialised bundle with a manifest, an inference function that validates what it is given, an applicability-domain verdict and a conformal interval attached to every prediction, a <code>shiny</code> application and a <code>plumber</code> endpoint whose complete source is printed and explained here, and a model card that states — in writing, before anyone asks — what the model is for, what it was trained on, how well it does under an honest split, and where it must not be used. The central experiment is the one that justifies all that machinery: what a deployed predictor does when it is handed a molecule from a chemistry it has never seen, and how confidently it does it.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad convierte un modelo en una herramienta utilizable por terceros. Se entrena el predictor de ROCK1 con una partición por andamiajes en tres bloques, se define su dominio de aplicabilidad mediante la distancia de Tanimoto a los cinco vecinos más cercanos y se construye a mano un predictor conformal inductivo, cuyo índice de cuantil se deduce en papel y se confirma con código. Se serializa el paquete del modelo con <code>saveRDS</code> junto a un manifiesto de versiones y sumas de verificación, mostrando dos trampas propias de R: el entorno que una clausura arrastra al archivo y la ausencia de información sobre las versiones de los paquetes. Se valida la entrada en tiempo de inferencia mostrando cada clase de rechazo, se escribe una función de predicción que devuelve potencia, intervalo, veredicto de dominio, vecinos más cercanos y atribuciones por ablación de bits, y se mide el fallo más importante de una herramienta desplegada: la extrapolación silenciosa. Se presentan, comentadas, una aplicación <code>shiny</code> —cuya función servidora se ejercita aquí con <code>testServer</code>, sin levantar ningún servidor— y una interfaz <code>plumber</code> con sus esquemas de petición y respuesta, además de una tarjeta de modelo y una de datos completas y las obligaciones de despliegue, registro, versionado, mantenimiento y retirada.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Serialise a trained model safely (<code>saveRDS</code>, version pinning, a manifest) and document it with a model card: intended use, data, metrics, limitations, applicability domain.</li>
<li>Build a small interactive application in which a chemist submits a structure and obtains a prediction with its uncertainty and an explanation, and test its server logic without launching a server.</li>
<li>Expose the same model as a REST API with a declared request and response schema, and know how such an endpoint is tested.</li>
<li>Deploy on a free tier with secrets handled correctly, and accept the maintenance and sunset obligations that deployment creates.</li>
<li>Recognise and prevent silent extrapolation: a confident number returned for a molecule outside the model's domain.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk in order; the model is retrained here in about a second, so nothing is read from another unit. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","Matrix","ranger","shiny","jsonlite","digest"))</code> once before the setup chunk. A web application and an HTTP server cannot run inside a document that must terminate, so the <code>shiny</code> and <code>plumber</code> code is written out in full and explained; the <code>shiny</code> server function is then exercised with <code>shiny::testServer()</code>, which drives its reactives in-process without opening a port, and the <code>plumber</code> handler is called directly. Every number quoted in the text comes from those calls. The applicability domain and the conformal predictor follow <a href="../U17_Uncertainty_AD/index.html">Unit 17</a>, in a compact implementation repeated here so that the unit stands alone.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(ranger); library(ggplot2)
  library(patchwork);  library(knitr);  library(jsonlite); library(digest); library(shiny)
})
set.seed(0)
PAL   <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
OKABE <- c("#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
dp  <- function(f) file.path("..", "data", f)
TMP <- tempdir()
MODEL_VERSION <- "1.0.0"
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ranger", as.character(packageVersion("ranger")),
    "| Matrix", as.character(packageVersion("Matrix")),
    "| shiny", as.character(packageVersion("shiny")),
    "| jsonlite", as.character(packageVersion("jsonlite")), "\n")



## 1. What changes when somebody else can press the button

Up to this unit the model has had one user, who knows how it was built, which compounds it saw, and what its error looks like on new scaffolds. Deployment removes all three. The person who submits a structure to a web page sees a number with two decimal places and a clean interface, and has no way of knowing that the molecule they submitted is unlike anything in the training set — unless the tool tells them.

That asymmetry is the whole subject of this unit. Everything built below exists to transmit, through the interface, the three things the author knows and the user does not: how certain the prediction is, whether the molecule is in the region where the model has evidence, and what the model is looking at when it answers.

<div class="box bridge"><h4>Domain context — who uses a published model, and how it goes wrong</h4>
<p>A published QSAR model has three kinds of user. A <strong>medicinal chemist</strong> ranking a handful of synthesis candidates wants one number per molecule and an honest flag; they will not install R. A <strong>computational colleague</strong> screening a catalogue of 200,000 compounds wants an interface they can call from a script — a REST API or the package itself — and will care about throughput. A <strong>referee or a future student</strong> wants to reproduce a figure, which means the model file, its version and the code that made it. Tools built for the first group but used by the second are the standard failure: a web form that silently truncates, times out, or — worst — happily returns predictions for compounds far outside the chemistry it was trained on. Published predictors of solubility, toxicity and potency are routinely used on molecules their authors never imagined, and because a regression model always returns <em>something</em>, nothing in the output says so. There is also a legal dimension: when users upload their own structures, those structures are confidential pre-publication intellectual property, and in a European or Chilean institutional context uploading a collaborator's unpublished compound to a third-party server can breach both an agreement and a data-protection policy. The group's rule is that a hosted tool states what it stores, stores nothing by default, and offers a local option for anything confidential.</p></div>

Models published with a usable interface are also the ones that get used: the deep-learning antibiotic screen that produced halicin is cited as much for its shared model and predictions as for the molecule<sup class="c"><a href="#r28">[28]</a></sup>. The literature on deploying machine learning is equally blunt about where projects fail: not in the modelling, but in everything around it — data validation at inference, versioning, monitoring and the slow drift of the environment<sup class="c"><a href="#r4">[4]</a><a href="#r5">[5]</a></sup>, which is why best-practice guidance for chemistry now treats deployment and reporting as part of the method<sup class="c"><a href="#r7">[7]</a></sup>. The reporting instruments that make a model usable by others — the **model card**<sup class="c"><a href="#r1">[1]</a></sup> and the **dataset card** or datasheet<sup class="c"><a href="#r2">[2]</a></sup> — were introduced precisely because the information that prevents misuse is known at training time and lost by publication time.

## 2. The thing being shipped

A deployed predictor is not a model object. It is a contract: given a structure, return a potency estimate, an interval that covers the truth at a stated rate, a verdict on whether the molecule lies in the model's domain, and enough explanation for a chemist to decide whether to believe it. The model is a random forest<sup class="c"><a href="#r24">[24]</a></sup> fitted with `ranger`<sup class="c"><a href="#r25">[25]</a></sup> on the curated ChEMBL table of Unit 6<sup class="c"><a href="#r14">[14]</a></sup>, split by scaffold rather than at random because a random split rewards memorisation of chemotypes and overstates what a model will do on new chemistry<sup class="c"><a href="#r21">[21]</a><a href="#r22">[22]</a></sup>. The next two chunks build all four parts on a three-way scaffold split — training, calibration (for the conformal intervals), and test (never touched until the evaluation).

<div class="box bridge"><h4>Substitution — featurisation in a deployed R service</h4>
<p>R has no RDKit binding in this environment, so the service built here cannot turn an arbitrary SMILES string into a Morgan fingerprint. Fingerprints come from the frozen sparse files <code>data/rock1_morgan_sparse.csv.gz</code> and <code>data/screening_morgan_sparse.csv.gz</code> (see <a href="../../data/manifest_fingerprints.json">manifest_fingerprints.json</a>), and the deployed prediction function therefore <em>looks up</em> the fingerprint of a submitted structure and refuses, with a clear message, any structure it cannot featurise. In a production R service that step is a call to a chemistry toolkit — ChemmineR, rcdk, or a small Python sidecar — and everything else in this unit is unchanged. One consequence is visible later: the "molecules from another chemistry" of the Python edition (table sugar, caffeine, a tripeptide) have no frozen fingerprints, so their role in the extrapolation experiment of Section 6 is played by the most dissimilar decoys of the screening library, which are real molecules measured against unrelated targets.</p></div>



In [ ]:
df  <- fread(dp("rock1_chembl_curated.csv"))
ex  <- df[censored == FALSE]
DESC <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
sp  <- fread(dp("rock1_morgan_sparse.csv.gz")); keep <- sp[id %in% ex$inchikey]
FP  <- sparseMatrix(i = match(keep$id, ex$inchikey), j = keep$bit + 1L, x = 1,
                    dims = c(nrow(ex), 2048))
X   <- cbind(as.matrix(FP), as.matrix(ex[, ..DESC]))
colnames(X) <- c(paste0("b", 0:2047), DESC); storage.mode(X) <- "double"
y <- ex$pActivity; scaf <- ex$scaffold_smiles
grp <- split(seq_along(scaf), scaf); ord <- order(-lengths(grp), names(grp))
n_tr <- floor(0.6 * length(scaf)); n_ca <- floor(0.2 * length(scaf))
TR <- CA <- TE <- integer(0)
for (k in ord) {
  i <- grp[[k]]
  if (length(TR) < n_tr) TR <- c(TR, i) else if (length(CA) < n_ca) CA <- c(CA, i) else TE <- c(TE, i)
}
cat(sprintf("%d compounds with exact measurements, %d scaffolds\n", nrow(ex), uniqueN(scaf)))
cat(sprintf("train %d | calibration %d | test %d; scaffolds shared between train and test: %d\n",
            length(TR), length(CA), length(TE), length(intersect(scaf[TR], scaf[TE]))))
cat(sprintf("target: pActivity, mean %.2f, sd %.2f on the training set\n", mean(y[TR]), sd(y[TR])))


In [ ]:
t0   <- Sys.time()
SEED <- 7L                        ## not 0: ranger reads seed = 0 as "no seed" (measured in Section 3)
fit <- ranger(x = X[TR, ], y = y[TR], num.trees = 300, seed = SEED, num.threads = 4,
              importance = "impurity")
pred_ca <- predict(fit, X[CA, ])$predictions
pred_te <- predict(fit, X[TE, ])$predictions
rmse_te <- sqrt(mean((pred_te - y[TE])^2)); mae_te <- mean(abs(pred_te - y[TE]))
r2_te   <- 1 - sum((pred_te - y[TE])^2) / sum((y[TE] - mean(y[TE]))^2)
resid_ca <- abs(pred_ca - y[CA])
conformal_q <- function(residuals, confidence) {       ## the (n+1)(1-eps) order statistic
  n <- length(residuals); k <- ceiling((n + 1) * confidence)
  list(q = sort(residuals)[min(k, n)], k = k, n = n)
}
CONF <- c(0.80, 0.90, 0.95)
QS   <- setNames(vapply(CONF, function(c) conformal_q(resid_ca, c)$q, numeric(1)), as.character(CONF))
q90  <- QS[["0.9"]]
FP_TR <- FP[TR, ]; SIZE_TR <- rowSums(FP_TR)             ## applicability domain, by sparse algebra
tanimoto_to_train <- function(q) {
  inter <- as.numeric(FP_TR %*% q)
  inter / (SIZE_TR + sum(q) - inter)
}
ad_distance <- function(q, k = 5) 1 - mean(sort(tanimoto_to_train(q), decreasing = TRUE)[seq_len(k)])
set.seed(0); probe <- sample(TR, 400)
d_train <- vapply(probe, function(i) {
  s <- sort(tanimoto_to_train(FP[i, ]), decreasing = TRUE)[2:6]   ## drop the compound itself
  1 - mean(s) }, numeric(1))
AD_THRESHOLD <- unname(quantile(d_train, 0.95))
d_test <- vapply(TE, function(i) ad_distance(FP[i, ]), numeric(1))
inside <- d_test <= AD_THRESHOLD
cat(sprintf("fitted in %.1f s\n", as.numeric(difftime(Sys.time(), t0, units = "secs"))))
cat(sprintf("scaffold test set: RMSE %.3f, MAE %.3f, R2 %.3f (n = %d)\n",
            rmse_te, mae_te, r2_te, length(TE)))
cq <- conformal_q(resid_ca, 0.90)
cat(sprintf("conformal 90%%: quantile index k = %d of n = %d calibration residuals -> half-width %.3f log units\n",
            cq$k, cq$n, cq$q))
cat(sprintf("applicability-domain threshold (95th percentile of training distances): %.3f\n", AD_THRESHOLD))
cat(sprintf("test compounds inside the domain: %d of %d (%.0f%%)\n",
            sum(inside), length(TE), 100 * mean(inside)))



Before going further, the conformal quantile deserves to be worked out by hand, because it is the one place in this pipeline where an off-by-one error silently costs coverage. Inductive conformal prediction<sup class="c"><a href="#r17">[17]</a><a href="#r18">[18]</a></sup> is the standard way of attaching a guaranteed error rate to a QSAR prediction, and has been proposed as a transparent replacement for ad-hoc applicability-domain rules<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup>.

With \( n \) calibration residuals and a target confidence \( 1-\varepsilon = 0.90 \), the index to take is

\[ k = \lceil (n+1)(1-\varepsilon) \rceil, \]

and the interval half-width is the \( k \)-th smallest residual. The \( +1 \) is the test compound itself: validity comes from the rank of its residual among the \( n+1 \) residuals being uniform under exchangeability.

| quantity | value |
|---|---|
| calibration compounds \( n \) | 552 |
| confidence \( 1-\varepsilon \) | 0.90 |
| \( (n+1)(1-\varepsilon) \) | \( 553 \times 0.9 = 497.7 \) |
| \( k = \lceil 497.7 \rceil \) | 498 |
| half-width | the 498th smallest of the 552 sorted residuals |

Two consequences follow immediately from that formula. The index cannot exceed \( n \), so \( n \geq 1/\varepsilon - 1 \): with fewer than 19 calibration points a 95 % interval is impossible, and the honest response is to report a lower confidence. And the quantile must come from data the model never saw — fitting it on the training residuals gives an interval that is far too narrow, because a random forest fits its training data almost perfectly.



In [ ]:
k_hand <- ceiling((length(resid_ca) + 1) * 0.90)
cat(sprintf("hand calculation: n = %d, (n+1)(1-eps) = %.1f, k = %d; code gave k = %d -> %s\n",
            length(resid_ca), (length(resid_ca) + 1) * 0.9, k_hand, cq$k,
            if (k_hand == cq$k) "agree" else "MISMATCH"))
err_te <- abs(pred_te - y[TE])
coverage <- do.call(rbind, lapply(CONF, function(c) {
  cc <- conformal_q(resid_ca, c)
  data.frame(confidence = c, k = cc$k, half_width = round(cc$q, 3),
             empirical_all = round(mean(err_te <= cc$q), 3),
             empirical_inside_AD = round(mean(err_te[inside] <= cc$q), 3),
             empirical_outside_AD = round(mean(err_te[!inside] <= cc$q), 3))
}))
kable(coverage, row.names = FALSE)
cat("\nnominal coverage is the 'confidence' column; the guarantee assumes calibration and test\n")
cat("compounds are exchangeable, which a scaffold split deliberately breaks\n")



The coverage table is the honest version of "the model predicts potency to within one log unit". Under a scaffold split the empirical coverage falls two to three percentage points short of nominal at every level — the exchangeability assumption behind the conformal guarantee is violated by construction, exactly as Unit 17 showed — and the two conditional columns say something less comfortable: the shortfall is not concentrated in the compounds the applicability domain flags, which are covered at a rate close to the rest. The flag is therefore not a refinement of the interval here. Its value lies elsewhere, and Section 6 measures where.

## 3. Serialising the bundle: saveRDS, versions and hashes

`saveRDS()` writes one R object to one file. It is the right tool, and it has two traps that cost laboratories real time.

The first is that **the file records nothing about the packages that produced it**. A `ranger` object is a list whose class is `ranger`; `predict()` on it dispatches to whatever version of `ranger` is installed when it is loaded, and if the internal structure changed between versions the result is one of three things: it works; it raises an error; or — the dangerous one — it loads and behaves differently. R is more forgiving than Python's pickle here (an `.rds` does not execute arbitrary code on load, and the object does not reference classes by import path), but the version question is identical.

The second is specific to R and catches everyone once: **a function carries its environment**. Saving a closure — a prediction function that captures the training data, or a model fitted inside a function that also held a large object — writes that whole environment into the file. The chunk below measures both the bundle and the trap.



In [ ]:
BUNDLE <- list(
  version = MODEL_VERSION, model = fit, descriptors = DESC,
  fp = list(radius = 2, bits = 2048, source = "frozen RDKit file, see manifest_fingerprints.json"),
  ad = list(measure = "1 - mean Tanimoto to the 5 nearest training compounds",
            threshold = AD_THRESHOLD, train_fp = FP_TR, train_size = SIZE_TR),
  conformal = as.list(QS),
  train = list(smiles = ex$std_smiles[TR], pActivity = y[TR], inchikey = ex$inchikey[TR]),
  metrics = list(RMSE = round(rmse_te, 3), MAE = round(mae_te, 3), R2 = round(r2_te, 3),
                 n_test = length(TE), split = "Bemis-Murcko scaffold, three-way, deterministic"),
  environment = list(R = paste(R.version$major, R.version$minor, sep = "."),
                     ranger = as.character(packageVersion("ranger")),
                     Matrix = as.character(packageVersion("Matrix")),
                     data.table = as.character(packageVersion("data.table"))))
model_path <- file.path(TMP, sprintf("rock1_rf_v%s.rds", MODEL_VERSION))
saveRDS(BUNDLE, model_path, compress = "xz")
manifest <- list(
  name = "rock1-potency-rf", version = MODEL_VERSION, created = "2026-10-18",
  task = "regression of pActivity (-log10 M) for human ROCK1",
  training_data = list(file = "rock1_chembl_curated.csv", source = "ChEMBL 37, CHEMBL3231",
                       sha256 = substr(digest(dp("rock1_chembl_curated.csv"), algo = "sha256", file = TRUE), 1, 32),
                       n_train = length(TR), n_calibration = length(CA), n_test = length(TE),
                       split = "Bemis-Murcko scaffold, deterministic"),
  features = list(fingerprint = "Morgan radius 2, 2048 bits (frozen file)",
                  descriptors = DESC, n_features = ncol(X)),
  hyperparameters = list(num.trees = 300, seed = SEED, others = "ranger defaults"),
  metrics_scaffold_test = BUNDLE$metrics,
  environment = BUNDLE$environment,
  artefact = list(file = basename(model_path),
                  sha256 = substr(digest(model_path, algo = "sha256", file = TRUE), 1, 32),
                  size_mb = round(file.size(model_path) / 1e6, 2)),
  licence = list(code = "MIT", model_and_data = "CC BY-SA 3.0 (inherited from ChEMBL)"))
write_json(manifest, file.path(TMP, "manifest.json"), auto_unbox = TRUE, pretty = TRUE)
cat(substr(toJSON(manifest, auto_unbox = TRUE, pretty = TRUE), 1, 1250), "...\n")
cat(sprintf("\nmodel file %s: %.2f MB, sha256 %s\n", basename(model_path),
            manifest$artefact$size_mb, manifest$artefact$sha256))


In [ ]:
restored <- readRDS(model_path)
same <- all.equal(predict(fit, X[TE[1:5], ])$predictions,
                  predict(restored$model, X[TE[1:5], ])$predictions)
cat("round trip identical:", isTRUE(same), "\n")
environment_check <- function(recorded, strict = FALSE) {
  current <- list(R = paste(R.version$major, R.version$minor, sep = "."),
                  ranger = as.character(packageVersion("ranger")),
                  Matrix = as.character(packageVersion("Matrix")),
                  data.table = as.character(packageVersion("data.table")))
  bad <- names(recorded)[vapply(names(recorded), function(k) !identical(current[[k]], recorded[[k]]),
                                logical(1))]
  msg <- vapply(bad, function(k) sprintf("%s: built with %s, running %s", k, recorded[[k]], current[[k]]),
                character(1))
  if (length(msg) && strict) stop("environment mismatch: ", paste(msg, collapse = "; "), call. = FALSE)
  unname(msg)
}
cat("environment check now:",
    if (!length(environment_check(BUNDLE$environment))) "identical to the recorded environment"
    else environment_check(BUNDLE$environment), "\n")
pretend <- modifyList(BUNDLE$environment, list(ranger = "0.14.1"))
cat("against a pretend older build:", environment_check(pretend), "\n")
cat("strict mode refuses to serve:",
    tryCatch(environment_check(pretend, strict = TRUE), error = conditionMessage), "\n")
BIG <- matrix(runif(2e6), ncol = 20)                     ## 16 MB that has nothing to do with the model
make_predictor_bad  <- function() { junk <- BIG; function(z) predict(fit, z)$predictions }
make_predictor_good <- local({ f <- fit; function() function(z) predict(f, z)$predictions })
p_bad  <- file.path(TMP, "closure_bad.rds");  saveRDS(make_predictor_bad(),  p_bad)
p_good <- file.path(TMP, "closure_good.rds"); saveRDS(make_predictor_good(), p_good)
cat(sprintf("\na closure that captured an unrelated 16 MB object: %.1f MB on disk\n", file.size(p_bad) / 1e6))
cat(sprintf("the same function with only the model in scope:     %.1f MB on disk\n", file.size(p_good) / 1e6))
rm(BIG); invisible(gc())



The environment check costs four lines and prevents the quietest class of production bug: a server that keeps answering after an unattended package upgrade changed the meaning of the model. The group's rule is that the serving environment is pinned (an `renv.lock` or a container image, as in Unit 25), the manifest is compared at start-up, and a mismatch is a refusal in production and a warning in development.

The closure measurement is the R-specific lesson. The two functions do exactly the same thing, and the file sizes differ by the size of an object that was merely *visible* where the function was defined. The rules that follow are mechanical: save data, not closures; if you must save a function, define it with `local()` so that its environment contains only what it needs; and check the size of every artefact you write, because an unexplained file size is an environment you did not mean to ship.

One more thing has to be true before a manifest means anything: the fit it describes must be repeatable. The chunk below fits the same forest twice, on a subset for speed, and compares the predictions.



In [ ]:
sub_tr <- TR[seq_len(400)]
twice  <- function(sd) {
  a <- ranger(x = X[sub_tr, ], y = y[sub_tr], num.trees = 100, seed = sd, num.threads = 1)
  b <- ranger(x = X[sub_tr, ], y = y[sub_tr], num.trees = 100, seed = sd, num.threads = 1)
  max(abs(predict(a, X[TE[1:50], ])$predictions - predict(b, X[TE[1:50], ])$predictions))
}
cat(sprintf("two fits with seed = 0:  largest difference in prediction %.3f log units\n", twice(0L)))
cat(sprintf("two fits with seed = %d:  largest difference in prediction %.3f log units\n", SEED, twice(SEED)))



`ranger` reads `seed = 0` as *no seed*: the forest is then seeded from the clock and two runs of the same script disagree by the amount printed above. Any non-zero value is honoured, and the second line is exactly zero. The Python edition's convention of writing `random_state=0` therefore does not transfer; this unit, and the manifest it writes, use `SEED` with a non-zero value, which is why the fitted model is recorded as reproducible rather than merely described. Note also that `set.seed()` governs R's own generator — the scaffold split, the sampling of probe compounds — and has no effect on the forest, which draws from the generator inside the C++ code.

R has no widely used equivalent of ONNX for this kind of model. The alternatives worth knowing are `qs` or `arrow` for faster and smaller serialisation of plain data, and the `vetiver` framework, which wraps a fitted model together with its metadata and generates the serving code — a package the group has not adopted but which solves exactly the problems this section works around by hand.

## 4. Validating the input before the model sees it

The featurisation above assumed a valid, standardised structure. A deployed tool receives whatever a user submits: a name instead of a SMILES, a salt, a mixture, an inorganic species, a protein-sized molecule, an empty string, ten thousand characters of pasted spreadsheet. Every one of those must produce a clear refusal rather than a prediction or a traceback.



In [ ]:
ORGANIC <- c("B", "C", "N", "O", "P", "S", "F", "I", "b", "c", "n", "o", "p", "s")
BRACKET_OK <- c("nH", "NH", "N", "O", "S", "C", "c", "n", "o", "s", "Si", "H", "CH", "CH2",
                "CH3", "NH2", "NH3", "OH", "SH", "C@", "C@@", "C@H", "C@@H", "N@", "N@@")
MAX_LEN <- 600L; MIN_HEAVY <- 3L; MAX_HEAVY <- 70L
FP_INDEX <- data.table(smiles = ex$std_smiles, inchikey = ex$inchikey, row = seq_len(nrow(ex)))
validate_structure <- function(smiles) {
  if (!is.character(smiles) || length(smiles) != 1L || is.na(smiles) || !nzchar(trimws(smiles)))
    return(list(ok = FALSE, reason = "empty input"))
  s <- trimws(smiles)
  if (nchar(s) > MAX_LEN)
    return(list(ok = FALSE, reason = sprintf("input longer than %d characters", MAX_LEN)))
  chars <- strsplit(s, "")[[1]]
  if (sum(chars == "(") != sum(chars == ")") || sum(chars == "[") != sum(chars == "]"))
    return(list(ok = FALSE, reason = "unbalanced brackets: not a parsable SMILES string"))
  note <- ""
  if (grepl(".", s, fixed = TRUE)) {                 ## mixture or salt: keep the largest fragment
    parts <- strsplit(s, ".", fixed = TRUE)[[1]]
    s <- parts[which.max(nchar(parts))]
    note <- "mixture or salt: largest fragment kept"
  }
  inside  <- gsub("[^A-Za-z@]", "", regmatches(s, gregexpr("\\[[^]]*\\]", s))[[1]])
  unknown <- setdiff(inside, BRACKET_OK)
  if (length(unknown))
    return(list(ok = FALSE, reason = sprintf("unsupported elements: %s",
                                             paste(unique(unknown), collapse = ", "))))
  plain <- gsub("Cl|Br", "", gsub("\\[[^]]*\\]", "", s))   ## atoms written without brackets
  if (length(setdiff(unique(strsplit(gsub("[^A-Za-z]", "", plain), "")[[1]]), ORGANIC)))
    return(list(ok = FALSE, reason = "not a parsable SMILES string (unknown atom symbols)"))
  heavy <- sum(strsplit(gsub("Cl|Br", "A", gsub("\\[[^]]*\\]", "X", s)), "")[[1]] %in%
               c(ORGANIC, "A", "X"))
  if (heavy < MIN_HEAVY || heavy > MAX_HEAVY)
    return(list(ok = FALSE, reason = sprintf("about %d heavy atoms, outside the supported range %d-%d",
                                             heavy, MIN_HEAVY, MAX_HEAVY)))
  hit <- FP_INDEX[smiles == s]
  if (!nrow(hit))
    return(list(ok = FALSE, reason = paste("structure has no precomputed fingerprint in this build;",
                                           "a production service calls a chemistry toolkit here")))
  list(ok = TRUE, smiles = s, row = hit$row[1], inchikey = hit$inchikey[1], heavy = heavy, note = note)
}
CASES <- list(c(ex$std_smiles[TE[1]], "a test-set kinase inhibitor"),
              c(paste0(ex$std_smiles[TE[2]], ".[Na+]"), "the same chemistry as a sodium salt"),
              c("fasudil", "a name, not a SMILES"), c("c1ccccc1C(", "malformed SMILES"),
              c("[Pt](Cl)(Cl)([NH3])[NH3]", "an inorganic complex"),
              c(strrep("C", 120), "a very large aliphatic chain"), c("   ", "blank input"),
              c("CC(=O)Oc1ccccc1C(=O)O", "aspirin: valid, but not in the frozen fingerprint file"))
for (case in CASES) {
  v <- validate_structure(case[1])
  cat(sprintf("%-46s %-9s %s\n", case[2], if (isTRUE(v$ok)) "accepted" else "REJECTED",
              if (isTRUE(v$ok))
                paste0(substr(v$smiles, 1, 34), if (nzchar(v$note)) paste0("  [", v$note, "]") else "")
              else v$reason))
}



Three design decisions in that function are worth defending, because each corresponds to a complaint the group has received about somebody else's tool.

**It never raises.** Every failure returns a reason string that the interface can show. A traceback on a web page is both unhelpful and an information leak.

**It standardises explicitly and says so.** When a salt is submitted, the tool keeps the largest fragment, predicts for the parent, and reports that it did — silently predicting for something other than what the user submitted is the shortest route to a wrong decision in a laboratory.

**It bounds the input.** Element and size limits are not pedantry: the training set contains small organic molecules with a dozen common elements, so a platinum complex or a 120-carbon chain is outside the chemistry the model saw, and refusing is more useful than extrapolating. These bounds, with the distance measure defined above, are the two layers of applicability domain the QSAR literature distinguishes: a coarse range box and a fine distance-to-training-set criterion<sup class="c"><a href="#r19">[19]</a><a href="#r20">[20]</a></sup>.

The last case is the one this R build adds: a perfectly valid molecule that the service cannot featurise. Refusing it is the correct behaviour for a tool whose featurisation step is a lookup, and the message says so rather than pretending the structure was wrong.

## 5. The prediction function the interfaces call

Both the application and the API are thin wrappers around a single function. Writing it once, here, is what keeps the two interfaces consistent — and what makes the contract testable.

The per-prediction explanation needs a word of its own. TreeSHAP is not available in R for `ranger` models, and the Monte-Carlo Shapley of `iml` must evaluate the model once per feature per sample: over 2,059 features that is minutes per molecule, measured below, which no interactive interface can pay. The contract therefore uses **bit ablation**: for each fingerprint bit that the molecule sets, predict again with that bit switched off, and report the change. It is a counterfactual attribution rather than a Shapley value — it ignores interactions between bits — and it costs one batched prediction.



In [ ]:
predict_compound <- function(smiles, confidence = 0.90, n_neighbours = 3, explain = TRUE,
                             bundle = BUNDLE) {
  v <- validate_structure(smiles)
  if (!isTRUE(v$ok))
    return(list(status = "rejected", reason = v$reason, input = smiles,
                model_version = bundle$version))
  row <- v$row
  features <- X[row, , drop = FALSE]
  value <- unname(predict(bundle$model, features)$predictions)
  half  <- bundle$conformal[[as.character(confidence)]]
  q     <- FP[row, ]
  sims  <- tanimoto_to_train(q)
  dist  <- 1 - mean(sort(sims, decreasing = TRUE)[1:5])
  near  <- order(sims, decreasing = TRUE)[seq_len(n_neighbours)]
  out <- list(
    status = "ok", model_version = bundle$version, input = smiles,
    standard_smiles = v$smiles, note = v$note,
    pActivity = round(value, 2),
    interval = round(c(value - half, value + half), 2),
    confidence = confidence,
    IC50_nM_estimate = round(10^(9 - value), 1),
    applicability = list(distance = round(dist, 3), threshold = round(bundle$ad$threshold, 3),
                         in_domain = dist <= bundle$ad$threshold),
    nearest_training_compounds = lapply(near, function(i)
      list(smiles = bundle$train$smiles[i], tanimoto = round(sims[i], 3),
           measured_pActivity = round(bundle$train$pActivity[i], 2))))
  if (isTRUE(explain)) {
    set_bits <- which(features[1, 1:2048] > 0)
    if (length(set_bits)) {
      ablated <- features[rep(1, length(set_bits)), , drop = FALSE]
      ablated[cbind(seq_along(set_bits), set_bits)] <- 0
      delta <- value - unname(predict(bundle$model, ablated)$predictions)
      top <- order(abs(delta), decreasing = TRUE)[seq_len(min(5, length(delta)))]
      out$top_features <- lapply(top, function(i)
        list(feature = paste0("bit_", set_bits[i] - 1L), contribution = round(delta[i], 3)))
    }
  }
  out
}
demo <- predict_compound(ex$std_smiles[TE[1]])
cat(substr(toJSON(demo, auto_unbox = TRUE, pretty = TRUE), 1, 1100), "...\n")
cat("\nrejected input: ", toJSON(predict_compound("fasudil"), auto_unbox = TRUE), "\n")


In [ ]:
t_one <- system.time(predict_compound(ex$std_smiles[TE[1]], explain = TRUE))[["elapsed"]]
t_noex <- system.time(predict_compound(ex$std_smiles[TE[1]], explain = FALSE))[["elapsed"]]
batch <- ex$std_smiles[TE[1:100]]
t_batch <- system.time(invisible(lapply(batch, predict_compound, explain = FALSE)))[["elapsed"]]
lat <- data.frame(what = c("one call\nwith explanation", "one call\nwithout explanation",
                           "100 calls\n(per call)"),
                  ms = c(1e3 * t_one, 1e3 * t_noex, 1e3 * t_batch / 100))
lat$what <- factor(lat$what, levels = lat$what)
print(ggplot(lat, aes(what, ms, fill = what)) + geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.0f ms", ms)), vjust = -0.4, size = 2.8) +
  scale_fill_manual(values = OKABE[c(2, 1, 3)]) + ylim(0, max(lat$ms) * 1.25) +
  labs(x = NULL, y = "milliseconds per compound", title = "Inference latency"))
cat(sprintf("single call with the explanation: %.0f ms | without: %.0f ms\n", 1e3 * t_one, 1e3 * t_noex))
cat(sprintf("100 compounds: %.1f s in total -> a 10,000-compound list would take about %.1f min\n",
            t_batch, 100 * t_batch / 60))
cat("conclusion: a web form is fine, a catalogue screen must use the package, not the interface\n")



<p class="caption"><b>Figure 1.</b> Latency of the prediction contract. The bit-ablation explanation adds one batched prediction over the molecule's set bits and is the dominant cost of a single call, which is why it is optional in the function signature and switched off for batches. The projection printed beneath the figure is the one that decides interface design: a form answering one molecule at a time is comfortable, and a colleague who wants to score a catalogue should be given the package and the model artefact rather than a loop over single requests.</p>



In [ ]:
shown <- order(pred_te)[round(seq(1, length(TE), length.out = 30))]
pdat <- data.frame(rank = seq_along(shown), pred = pred_te[shown], obs = y[TE][shown],
                   lo = pred_te[shown] - q90, hi = pred_te[shown] + q90,
                   domain = ifelse(inside[shown], "in domain", "outside domain"))
print(ggplot(pdat, aes(rank)) +
  geom_linerange(aes(ymin = lo, ymax = hi, colour = domain), linewidth = 0.7, alpha = 0.8) +
  geom_point(aes(pred, y = pred, colour = domain), size = 1.2) +
  geom_point(aes(y = obs), shape = 4, size = 1.6, colour = "black") +
  scale_colour_manual(values = c(`in domain` = OKABE[1], `outside domain` = OKABE[2]), name = NULL) +
  labs(x = "thirty test compounds, ordered by predicted potency", y = "pActivity",
       title = "Prediction, 90% conformal interval, and the measured value (cross)") +
  theme(axis.text.x = element_blank(), legend.position = "bottom"))
cat(sprintf("interval half-width %.2f log units; the measured value lies inside the interval for %.0f%% of these 30 compounds\n",
            q90, 100 * mean(abs(pred_te[shown] - y[TE][shown]) <= q90)))
cat(sprintf("of the 30, %d are inside the applicability domain and %d outside\n",
            sum(inside[shown]), sum(!inside[shown])))



<p class="caption"><b>Figure 2.</b> Thirty compounds of the scaffold-split test set as the deployed tool would report them: the point estimate, the 90 % conformal interval, and the measured value as a cross, with colour giving the applicability-domain verdict. The intervals are wide — twice the printed half-width, so about 2.7 log units from end to end — which is the honest consequence of calibrating on compounds whose scaffolds the model has never seen, and the crosses falling outside them are the coverage shortfall quantified in the table of Section 2.</p>

## 6. The failure that matters: silent extrapolation

Now the experiment. A regression model always returns a number. Ask a random forest trained on ROCK1 inhibitors about a molecule from an unrelated chemistry and it will answer with the same two decimal places it uses for a kinase inhibitor — because a forest averages the leaves its input falls into, and an unfamiliar molecule simply lands in leaves belonging to the training distribution. The prediction is not wrong by a little; it is uninformative, and it looks exactly like an informative one.

The chunk below asks the deployed model about four groups: test compounds inside the applicability domain, test compounds outside it, property-matched decoys from the screening library (molecules measured against unrelated targets), and the two hundred decoys whose maximum Tanimoto similarity to any ROCK1 ligand is lowest — the furthest chemistry for which this R build has fingerprints.



In [ ]:
lib <- fread(dp("screening_library.csv"))
spl <- fread(dp("screening_morgan_sparse.csv.gz")); keepl <- spl[id %in% lib$inchikey]
FP_LIB <- sparseMatrix(i = match(keepl$id, lib$inchikey), j = keepl$bit + 1L, x = 1,
                       dims = c(nrow(lib), 2048))
X_LIB <- cbind(as.matrix(FP_LIB), as.matrix(lib[, ..DESC])); storage.mode(X_LIB) <- "double"
colnames(X_LIB) <- colnames(X)
score_rows <- function(rows, fpm, xm) {
  preds <- unname(predict(fit, xm[rows, , drop = FALSE])$predictions)
  dists <- vapply(rows, function(i) ad_distance(fpm[i, ]), numeric(1))
  data.frame(pred = preds, dist = dists)
}
set.seed(0)
dec_rows <- which(lib$label == 0)
far_rows <- dec_rows[order(lib$max_tanimoto_to_rock1[dec_rows])][1:200]
groups <- list(
  "test, in domain"      = score_rows(TE[inside][1:min(120, sum(inside))], FP, X),
  "test, outside domain" = score_rows(TE[!inside][1:120], FP, X),
  "screening decoys"     = score_rows(sample(dec_rows, 200), FP_LIB, X_LIB),
  "most dissimilar decoys" = score_rows(far_rows, FP_LIB, X_LIB))
summary_tbl <- do.call(rbind, lapply(names(groups), function(nm) {
  g <- groups[[nm]]
  data.frame(group = nm, n = nrow(g), mean_prediction = round(mean(g$pred), 2),
             sd_prediction = round(sd(g$pred), 2), mean_AD_distance = round(mean(g$dist), 3),
             flagged_outside = sprintf("%.0f%%", 100 * mean(g$dist > AD_THRESHOLD)))
}))
kable(summary_tbl, row.names = FALSE)
cat(sprintf("\ntraining-set mean pActivity: %.2f (the value the forest reverts to)\n", mean(y[TR])))
far <- groups[["most dissimilar decoys"]]
cat(sprintf("the most dissimilar decoys receive potencies from %.2f to %.2f (IC50 %.0f to %.0f nM),\n",
            min(far$pred), max(far$pred), 10^(9 - max(far$pred)), 10^(9 - min(far$pred))))
cat(sprintf("with a spread of %.2f log units against %.2f for the in-domain test compounds\n",
            sd(far$pred), sd(groups[["test, in domain"]]$pred)))


In [ ]:
long <- do.call(rbind, lapply(names(groups), function(nm) data.frame(group = nm, groups[[nm]])))
long$group <- factor(long$group, levels = names(groups))
g1 <- ggplot(long, aes(group, pred, colour = group)) +
  geom_jitter(width = 0.18, size = 0.5, alpha = 0.45, show.legend = FALSE) +
  stat_summary(fun = mean, geom = "crossbar", width = 0.45, linewidth = 0.3, colour = "black") +
  geom_hline(yintercept = mean(y[TR]), linetype = 2, colour = "grey40") +
  annotate("text", x = 4.4, y = mean(y[TR]) + 0.12, hjust = 1, size = 2.3, colour = "grey30",
           label = "training mean") +
  scale_colour_manual(values = OKABE[c(1, 4, 2, 6)]) +
  scale_x_discrete(labels = function(x) gsub(", ", ",\n", x)) +
  labs(x = NULL, y = "predicted pActivity", title = "Predictions, by distance from the training set") +
  theme(axis.text.x = element_text(size = 6))
qs <- quantile(d_test, c(0, 0.25, 0.5, 0.75, 1))
qdat <- do.call(rbind, lapply(seq_len(4), function(i) {
  sel <- d_test >= qs[i] & d_test <= qs[i + 1]
  data.frame(bin = sprintf("%.2f-%.2f", qs[i], qs[i + 1]),
             rmse = sqrt(mean((pred_te[sel] - y[TE][sel])^2)), n = sum(sel))
}))
g2 <- ggplot(qdat, aes(bin, rmse)) + geom_col(fill = OKABE[1], width = 0.6) +
  geom_text(aes(label = sprintf("%.2f\n(n=%d)", rmse, n)), vjust = -0.2, size = 2.3) +
  geom_hline(yintercept = rmse_te, colour = OKABE[2], linetype = 2, linewidth = 0.4) +
  ylim(0, max(qdat$rmse) * 1.35) +
  labs(x = "applicability-domain distance (quartiles of the test set)", y = "RMSE (log units)",
       title = "Error against distance") + theme(axis.text.x = element_text(size = 6))
print(g1 | g2)
cat(sprintf("RMSE by AD-distance quartile: %s | overall %.3f\n",
            paste(round(qdat$rmse, 3), collapse = ", "), rmse_te))



<p class="caption"><b>Figure 3.</b> Silent extrapolation, measured. Left: predictions for four groups of molecules at increasing distance from the training set, with the group means as black bars and the training mean as a dashed line. As the chemistry becomes unfamiliar the spread of the predictions collapses towards the training mean, while the means stay comfortably inside the range a chemist would read as "weakly to moderately active" — and nothing in those numbers says they are meaningless. Right: the root-mean-square error of the test set by quartile of applicability-domain distance, with the overall test RMSE as a dashed line. The numbers printed above the chunk are the measured ones; what matters is how little the error is graded <em>within</em> a scaffold-split test set, where every compound already comes from an unseen chemotype. The flag earns its place by excluding the chemistry shown on the left, not by ranking the compounds the model was always going to be asked about.</p>

None of this means the model is bad: its error on the scaffold test set is within a small factor of the roughly half a log unit of experimental noise in the source data<sup class="c"><a href="#r23">[23]</a></sup>. The lesson is that **the output of a model carries no information about whether it should have been produced**, so the interface must carry it instead. Three mechanisms do that, and all three are in the prediction contract: the validation of Section 4 refuses what is obviously outside the chemistry, the applicability-domain distance flags what is subtly outside it, and the conformal interval widens the claim to a range whose coverage has been measured. An interface that shows the number and hides the other three is the thing this unit exists to prevent.

<div class="exercise"><h4>Exercise 1</h4>
<p>The applicability-domain threshold used here is the 95th percentile of the training compounds' own distances. Recompute the table of Section 6 for thresholds at the 75th and 99th percentiles, and decide which you would ship. State the decision as a trade-off: how many useful predictions does the strict threshold refuse, and how much error does the permissive one admit?</p></div>
## 7. The application: shiny, line by line

`shiny` splits an application into a **UI** object describing the page and a **server** function describing how outputs are computed from inputs. The server runs in a reactive graph: an expression wrapped in `reactive()` recomputes only when something it reads changes, and an `output$x <- render*()` block recomputes only when the reactives it reads change. That execution model is the opposite of Streamlit's re-run-everything, and it has one consequence worth internalising: expensive work belongs in a `reactive()` that depends on as little as possible.

The complete application is written below. It cannot be served from a document that must terminate, so the code is printed; the next chunk defines the same server function for real and drives it with `shiny::testServer()`.



In [ ]:
## app.R - the complete shiny application. Run with: shiny::runApp("app.R")
library(shiny)
library(rockserve)        ## the package of Unit 24, extended with predict_compound()

BUNDLE <- rockserve::load_bundle("artefacts/rock1_rf_v1.0.0.rds", strict = FALSE)

ui <- fluidPage(
  titlePanel("ROCK1 potency predictor"),
  tags$p(class = "lead", sprintf(
    paste("Random forest on Morgan fingerprints and physicochemical descriptors, model version %s.",
          "Scaffold-split test RMSE %.2f log units on %d compounds.",
          "Predictions are estimates, not measurements."),
    BUNDLE$version, BUNDLE$metrics$RMSE, BUNDLE$metrics$n_test)),
  sidebarLayout(
    sidebarPanel(
      width = 3,
      selectInput("confidence", "Confidence level", choices = c(0.80, 0.90, 0.95), selected = 0.90),
      checkboxInput("explain", "Show feature attributions", value = TRUE),
      tags$hr(),
      tags$p(tags$b("Privacy."), "Structures are processed in memory and never stored.",
             "For confidential compounds, run this app locally:",
             tags$code("remotes::install_github('<organisation>/rockserve')")),
      tags$hr(),
      tags$small(sprintf("model %s | app %s", BUNDLE$version, utils::packageVersion("rockserve")))),
    mainPanel(
      textInput("smiles", "SMILES string", width = "100%",
                value = "O=C(NCc1ccccc1)c1ccc(-c2cn[nH]c2)cc1"),
      fileInput("csv", "...or upload a CSV with a 'smiles' column", accept = ".csv"),
      uiOutput("verdict"),                 ## the applicability-domain banner, rendered first
      fluidRow(column(4, uiOutput("box_p")), column(4, uiOutput("box_ci")),
               column(4, uiOutput("box_ic50"))),
      tableOutput("neighbours"),
      plotOutput("attributions", height = "220px"),
      tableOutput("batch"),
      downloadButton("download", "Download predictions"),
      tags$hr(),
      tags$small("Model card: https://github.com/<organisation>/rockserve/blob/main/MODEL_CARD.md",
                 "| Code: MIT | Model and data: CC BY-SA 3.0 (ChEMBL) | Contact: the laboratory")))
)

server <- function(input, output, session) {
  result <- reactive({                      ## one reactive: everything else reads from it
    req(input$smiles)
    rockserve::predict_compound(input$smiles, bundle = BUNDLE,
                                confidence = as.numeric(input$confidence),
                                explain = isTRUE(input$explain))
  })
  output$verdict <- renderUI({
    r <- result()
    if (identical(r$status, "rejected"))
      return(div(class = "alert alert-danger", paste("Input rejected:", r$reason)))
    ad <- r$applicability
    if (isTRUE(ad$in_domain))
      div(class = "alert alert-success",
          sprintf("Inside the applicability domain (distance %.2f <= %.2f).",
                  ad$distance, ad$threshold))
    else
      div(class = "alert alert-danger",
          sprintf(paste("OUTSIDE the applicability domain (distance %.2f > %.2f).",
                        "This molecule is unlike the training compounds; the number below is an",
                        "extrapolation and should not be used to rank candidates."),
                  ad$distance, ad$threshold))
  })
  output$box_p    <- renderUI(req(result()$status == "ok") %then%
                                h3(sprintf("pActivity %.2f", result()$pActivity)))
  output$box_ci   <- renderUI(req(result()$status == "ok") %then%
                                h4(sprintf("%d%% interval %.2f - %.2f",
                                           round(100 * result()$confidence),
                                           result()$interval[1], result()$interval[2])))
  output$box_ic50 <- renderUI(req(result()$status == "ok") %then%
                                h4(sprintf("IC50 estimate %s nM",
                                           format(result()$IC50_nM_estimate, big.mark = ","))))
  output$neighbours <- renderTable({
    req(result()$status == "ok")
    do.call(rbind, lapply(result()$nearest_training_compounds, as.data.frame))
  })
  output$attributions <- renderPlot({
    req(isTRUE(input$explain), result()$status == "ok", !is.null(result()$top_features))
    att <- do.call(rbind, lapply(result()$top_features, as.data.frame))
    ggplot2::ggplot(att, ggplot2::aes(stats::reorder(feature, contribution), contribution,
                                      fill = contribution > 0)) +
      ggplot2::geom_col(show.legend = FALSE) + ggplot2::coord_flip() +
      ggplot2::labs(x = NULL, y = "change in predicted pActivity when the bit is removed")
  })
  batch <- reactive({                        ## batch mode: capped, no explanations
    req(input$csv)
    frame <- utils::read.csv(input$csv$datapath)
    validate(need("smiles" %in% names(frame), "The uploaded file needs a column named 'smiles'."))
    rows <- lapply(utils::head(frame$smiles, 500), rockserve::predict_compound,
                   bundle = BUNDLE, confidence = as.numeric(input$confidence), explain = FALSE)
    rockserve::as_table(rows)
  })
  output$batch <- renderTable(batch())
  output$download <- downloadHandler(
    filename = function() sprintf("rock1_predictions_%s.csv", Sys.Date()),
    content  = function(path) utils::write.csv(batch(), path, row.names = FALSE))
}

shinyApp(ui, server)



Seven things in that file are deliberate.

**One `reactive()` holds the prediction.** Every output reads `result()`, so the model is called once per change of input, not once per output. The `shiny` analogue of Streamlit's `cache_resource` is simpler: the bundle is loaded at the top level of `app.R`, which runs once per R process and is therefore shared by every session.

**The caption states the model's identity and error.** Version and scaffold-split RMSE appear above the input box, not in a document nobody opens. A user who sees "RMSE 0.9 log units" reads the prediction differently.

**The applicability-domain verdict is rendered first and as an alert.** Out-of-domain is a red banner above the number, with a sentence saying what it means and what not to do with it. This single element is the difference between a tool that misleads and one that does not.

**The input is echoed after standardisation.** What the table and the attributions describe is the parent structure the model actually scored; a user who submitted a salt sees that it was stripped.

**The evidence is shown, not just the answer.** The three nearest training compounds with their measured potencies let a chemist judge the prediction with their own expertise, which is more persuasive than any attribution plot.

**Batch mode is capped and explicit.** Five hundred rows, no explanations, a download button. The cap exists because a free-tier container has one core and a request timeout.

**The privacy statement is in the sidebar, where a user looking for it will look.** It says what happens to submitted structures and offers the local alternative.



In [ ]:
predictor_server <- function(input, output, session) {     ## the same logic, isolated for testing
  result <- reactive({
    req(input$smiles)
    predict_compound(input$smiles, confidence = as.numeric(input$confidence),
                     explain = isTRUE(input$explain))
  })
  verdict <- reactive({
    r <- result()
    if (identical(r$status, "rejected")) sprintf("REJECTED: %s", r$reason)
    else if (isTRUE(r$applicability$in_domain))
      sprintf("in domain (distance %.2f <= %.2f)", r$applicability$distance, r$applicability$threshold)
    else sprintf("OUTSIDE the domain (distance %.2f > %.2f): extrapolation",
                 r$applicability$distance, r$applicability$threshold)
  })
  output$headline <- renderText({
    r <- result()
    if (identical(r$status, "rejected")) verdict()
    else sprintf("pActivity %.2f  [%.2f, %.2f]  |  %s", r$pActivity, r$interval[1], r$interval[2],
                 verdict())
  })
}
in_domain_smiles <- ex$std_smiles[TE[inside][1]]
out_domain_smiles <- ex$std_smiles[TE[!inside][1]]
testServer(predictor_server, {
  session$setInputs(smiles = in_domain_smiles, confidence = "0.90", explain = FALSE)
  cat("1. a test compound inside the domain\n   ", output$headline, "\n")
  session$setInputs(smiles = out_domain_smiles)
  cat("2. a test compound outside the domain\n   ", output$headline, "\n")
  session$setInputs(smiles = "fasudil")
  cat("3. a compound name instead of a structure\n   ", output$headline, "\n")
  session$setInputs(smiles = in_domain_smiles, confidence = "0.95")
  cat("4. the same molecule at 95% confidence\n   ", output$headline, "\n")
})



`shiny::testServer()` runs the server function with a simulated session: `session$setInputs()` pushes values into the reactive graph and the outputs are read exactly as the browser would receive them — no port is opened and no browser is involved. The four interactions above are the four cases the application must get right, and they are now a test that can live in `tests/testthat/test-app.R` alongside the tests of Unit 24. Note the fourth: raising the confidence widens the interval, which is the behaviour a user will check first.



In [ ]:
boxes <- data.frame(
  x = c(5.0, 1.5, 6.3, 6.3, 3.6, 5.1, 6.6, 6.3, 4.4, 8.2, 6.3),
  y = c(6.3, 3.2, 5.5, 4.8, 3.9, 3.9, 3.9, 3.1, 1.9, 1.9, 0.7),
  w = c(9.6, 2.6, 7.0, 7.0, 2.2, 2.2, 2.2, 7.0, 3.3, 3.3, 7.0),
  h = c(0.8, 5.0, 0.6, 0.6, 0.8, 0.8, 0.8, 0.65, 1.2, 1.2, 0.7),
  fill = c("#e8eef7", "#f7f7f7", "#f3f6fa", "#f3f6fa", "#e6f4ea", "#e6f4ea", "#e6f4ea",
           "#fdeaea", "#f3f6fa", "#f3f6fa", "#f7f7f7"),
  col = c("#0072B2", "#999999", "#0072B2", "#0072B2", "#009E73", "#009E73", "#009E73",
          "#D55E00", "#0072B2", "#0072B2", "#999999"),
  label = c(sprintf("ROCK1 potency predictor  -  model v%s, scaffold-split RMSE %.2f log units",
                    MODEL_VERSION, rmse_te),
            "sidebar\n\nconfidence\n0.80 / 0.90 / 0.95\n\nshow attributions\n\nprivacy notice\nand local option",
            "SMILES input box", "upload a CSV with a 'smiles' column (max 500 rows)",
            sprintf("pActivity\n%.2f", demo$pActivity),
            sprintf("90%% interval\n%.1f - %.1f", demo$interval[1], demo$interval[2]),
            sprintf("IC50 estimate\n%.0f nM", demo$IC50_nM_estimate),
            "applicability-domain banner - rendered before the number",
            "nearest training compounds\nwith measured potency", "bit-ablation attributions",
            "model card | licence | contact | model and app versions"),
  size = c(3.0, 2.3, 2.6, 2.3, 2.6, 2.6, 2.6, 2.8, 2.3, 2.3, 2.3))
print(ggplot(boxes) +
  geom_rect(aes(xmin = x - w / 2, xmax = x + w / 2, ymin = y - h / 2, ymax = y + h / 2),
            fill = boxes$fill, colour = boxes$col, linewidth = 0.5) +
  geom_text(aes(x, y, label = label), size = boxes$size, lineheight = 1.05) +
  coord_cartesian(xlim = c(0, 10), ylim = c(0, 7)) + theme_void())
cat(sprintf("panel values taken from the contract: pActivity %.2f, interval [%.2f, %.2f], IC50 %.0f nM\n",
            demo$pActivity, demo$interval[1], demo$interval[2], demo$IC50_nM_estimate))
cat("the red band is the only element whose absence would make this tool dangerous\n")



<p class="caption"><b>Figure 4.</b> Layout of the application, drawn with the numbers the prediction contract returned for the example molecule. The prediction, its interval and the derived IC50 occupy one row; immediately beneath them, and at full width, sits the applicability-domain banner. Below that is the evidence a chemist can judge directly — the nearest training compounds with their <em>measured</em> potencies — and only then the attributions. The same layout decisions apply in any framework; what matters is the order, which is the order in which a hurried reader stops reading.</p>

## 8. The API: plumber, schemas and testing

An application serves people; an API serves programs. In R the standard tool is `plumber`, which turns annotated functions into HTTP endpoints: `#* @post /predict` above a function makes it a route, and the function's return value is serialised to JSON. `plumber` is **not installed in this environment**, so the service below is written out as non-evaluated source and explained; the handler function it wraps is then called directly, which is exactly what a test does.

Unlike FastAPI, `plumber` has no built-in schema validation — R has no type annotations the framework can read — so the validation that Pydantic would do declaratively has to be written explicitly. That is the one place where the R service is longer than its Python counterpart, and it is worth seeing why.



In [ ]:
## plumber.R - the complete service. Run with:
##   plumber::pr("plumber.R") |> plumber::pr_run(host = "0.0.0.0", port = 8000)
library(plumber)
library(rockserve)

MODEL_PATH <- Sys.getenv("ROCK1_MODEL", "artefacts/rock1_rf_v1.0.0.rds")
API_KEY    <- Sys.getenv("ROCK1_API_KEY", "")     ## secret from the environment, never in code
MAX_BATCH  <- 200L
BUNDLE     <- rockserve::load_bundle(MODEL_PATH, strict = TRUE)   ## fail at start-up, not later

## @apiTitle ROCK1 potency predictor
## @apiDescription Predicted pActivity with a conformal interval and an applicability-domain
##   verdict. Not for clinical use.
## @apiVersion 1.0.0

#* Reject the request when an API key is configured and not supplied
#* @filter auth
function(req, res) {
  open_route <- grepl("^/(health|model|__docs__|openapi)", req$PATH_INFO)
  if (nzchar(API_KEY) && !open_route &&
      !identical(req$HTTP_X_API_KEY, API_KEY)) {
    res$status <- 401
    return(list(error = "invalid or missing X-API-Key header"))
  }
  plumber::forward()
}

#* Liveness probe used by the hosting platform and by monitoring
#* @get /health
function() list(status = "ok", model_version = BUNDLE$version)

#* The manifest: what this service is serving, and how well it does
#* @get /model
function() list(version = BUNDLE$version, metrics = BUNDLE$metrics,
                applicability_domain = list(measure = BUNDLE$ad$measure,
                                            threshold = round(BUNDLE$ad$threshold, 3)),
                model_card = "https://github.com/<organisation>/rockserve/blob/main/MODEL_CARD.md")

#* Predict potency for up to 200 structures
#* @post /predict
#* @serializer unboxedJSON
function(req, res) {
  body <- req$body                                   ## validation written by hand: no schema engine
  problems <- character(0)
  if (is.null(body$smiles) || !is.character(body$smiles))
    problems <- c(problems, "'smiles' must be an array of strings")
  if (length(body$smiles) < 1L || length(body$smiles) > MAX_BATCH)
    problems <- c(problems, sprintf("'smiles' must hold between 1 and %d entries", MAX_BATCH))
  confidence <- if (is.null(body$confidence)) 0.90 else as.numeric(body$confidence)
  if (!confidence %in% c(0.80, 0.90, 0.95))
    problems <- c(problems, "'confidence' must be one of 0.8, 0.9, 0.95")
  if (length(problems)) {
    res$status <- 422                                 ## unprocessable entity, as FastAPI would
    return(list(error = "invalid request", detail = problems))
  }
  started <- Sys.time()
  results <- lapply(body$smiles, rockserve::predict_compound, bundle = BUNDLE,
                    confidence = confidence, explain = isTRUE(body$explain))
  elapsed <- as.numeric(difftime(Sys.time(), started, units = "secs")) * 1000
  n_rejected <- sum(vapply(results, function(r) identical(r$status, "rejected"), logical(1)))
  message(sprintf("predict n=%d rejected=%d elapsed_ms=%.0f version=%s",   ## counts, not chemistry
                  length(results), n_rejected, elapsed, BUNDLE$version))
  list(model_version = BUNDLE$version, n_requested = length(results), n_rejected = n_rejected,
       elapsed_ms = round(elapsed, 1), predictions = results)
}



The design decisions, in the order a reviewer would question them.

**The model is loaded at start-up, with `strict = TRUE`.** A service that starts successfully and fails on the first request is harder to diagnose than one that refuses to start. The environment check of Section 3 runs here.

**The request is validated explicitly, and the error is a 422 with a list of problems.** FastAPI gets this from type annotations; `plumber` does not, so the checks are written out — batch size, confidence level, the type of `smiles` — and they run before any chemistry. The `plumber` ecosystem can generate an OpenAPI specification from the annotations, but the *body* schema still has to be documented by hand.

**The response shape is fixed, including the rejected case.** A structure that fails validation returns `status: "rejected"` with a reason *inside* a 200 response rather than failing the whole batch.

**`/health` and `/model` exist and bypass the key filter.** The first is what the platform polls to decide whether the container is alive; the second is what a user — or a referee — calls to learn which version produced a number, with a link to the model card.

**The secret comes from the environment.** `Sys.getenv("ROCK1_API_KEY")` and a filter: no key in the source, in the repository or in the image. If no key is configured the service is open, which is a deliberate default for an internal tool and a deliberate decision to document.

**The log line records counts, not chemistry.** Structures are confidential; `n=12 rejected=1 elapsed_ms=380 version=1.0.0` is enough to monitor usage and latency without storing anyone's compounds.



In [ ]:
handle_predict <- function(body) {                    ## the handler, without the HTTP layer
  problems <- character(0)
  if (is.null(body$smiles) || !is.character(body$smiles))
    problems <- c(problems, "'smiles' must be an array of strings")
  else if (length(body$smiles) < 1L || length(body$smiles) > 200L)
    problems <- c(problems, "'smiles' must hold between 1 and 200 entries")
  confidence <- if (is.null(body$confidence)) 0.90 else as.numeric(body$confidence)
  if (!confidence %in% c(0.80, 0.90, 0.95))
    problems <- c(problems, "'confidence' must be one of 0.8, 0.9, 0.95")
  if (length(problems)) return(list(status = 422L,
                                    body = list(error = "invalid request", detail = problems)))
  started <- Sys.time()
  results <- lapply(body$smiles, predict_compound, confidence = confidence,
                    explain = isTRUE(body$explain))
  keep <- c("status", "input", "standard_smiles", "pActivity", "interval",
            "IC50_nM_estimate", "applicability", "reason")
  elapsed <- 1000 * as.numeric(difftime(Sys.time(), started, units = "secs"))
  list(status = 200L,
       body = list(model_version = MODEL_VERSION, n_requested = length(results),
                   n_rejected = sum(vapply(results, function(r) r$status == "rejected", logical(1))),
                   elapsed_ms = round(elapsed, 1),
                   predictions = lapply(results, function(r) r[intersect(keep, names(r))])))
}
request <- list(smiles = c(ex$std_smiles[TE[1]], lib$std_smiles[far_rows[1]], "fasudil"),
                confidence = 0.90, explain = FALSE)
cat("POST /predict  (X-API-Key: ...)\n\n")
cat(toJSON(request, auto_unbox = TRUE, pretty = TRUE), "\n\n")
resp <- handle_predict(request)
cat(sprintf("%d OK\n\n", resp$status))
cat(substr(toJSON(resp$body, auto_unbox = TRUE, pretty = TRUE, digits = 4), 1, 1500), "...\n")
bad <- handle_predict(list(smiles = c(ex$std_smiles[TE[1]]), confidence = 0.5))
cat(sprintf("\na request with an unsupported confidence level -> %d %s\n", bad$status,
            toJSON(bad$body, auto_unbox = TRUE)))
cat(sprintf("\nserver-side log line: predict n=%d rejected=%d elapsed_ms=%.0f version=%s\n",
            resp$body$n_requested, resp$body$n_rejected, resp$body$elapsed_ms, MODEL_VERSION))


In [ ]:
stages <- data.frame(x = c(1.0, 2.9, 4.8, 6.7, 8.6),
                     label = c("client\nPOST /predict", "request validation\n(written by hand)",
                               "API-key filter", "structure validation\n+ standardisation",
                               "model + conformal\n+ domain check"),
                     fill = c("#e8eef7", "#e8eef7", "#e8eef7", "#fff6e5", "#e6f4ea"))
exits <- data.frame(x = c(2.9, 4.8, 6.7, 8.6), y = 2.0,
                    label = c("422 invalid request", "401 unauthorised", "status: rejected",
                              "status: ok + interval\n+ domain verdict"),
                    colour = c("#D55E00", "#D55E00", "#D55E00", "#007a52"))
print(ggplot() +
  geom_rect(data = stages, aes(xmin = x - 0.85, xmax = x + 0.85, ymin = 3.3, ymax = 4.5),
            fill = stages$fill, colour = "#0072B2", linewidth = 0.5) +
  geom_text(data = stages, aes(x, 3.9, label = label), size = 2.5, lineheight = 1.05) +
  geom_segment(data = stages[1:4, ], aes(x = x + 0.9, xend = x + 1.15, y = 3.9, yend = 3.9),
               arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
  geom_segment(data = exits, aes(x = x, xend = x, y = 3.25, yend = y + 0.35),
               arrow = arrow(length = unit(0.1, "cm"), type = "closed"),
               colour = "grey60", linewidth = 0.3) +
  geom_text(data = exits, aes(x, y, label = label), size = 2.4, colour = exits$colour,
            lineheight = 1.05, vjust = 1) +
  annotate("text", x = 5, y = 5.1, size = 2.9, fontface = "italic",
           label = "every exit from the path is a documented response, never a traceback") +
  coord_cartesian(xlim = c(0, 10), ylim = c(0.8, 5.4)) + theme_void())
cat("the four exits are the four tests the service needs: bad payload, missing key,\n")
cat("unparsable structure, and a reference molecule whose prediction must not change\n")



<p class="caption"><b>Figure 5.</b> The path of a request through the service and its four possible exits. Validation happens in two layers — the request body first, the chemistry second — so a malformed request never reaches the featuriser and an unsubmittable structure never reaches the model. Each exit is a documented status code or status field, which is what makes the service testable: `testthat` calls the handler directly, as the chunk above does, and `plumber::pr()` plus `httr2` tests the HTTP layer separately.</p>

## 9. The model card and the dataset card

A **model card** is a short structured document describing what a model is for, what it was trained on, how it performs, and where it must not be used<sup class="c"><a href="#r1">[1]</a></sup>. It was proposed to prevent exactly the failure measured in Section 6: a model used outside the conditions in which it was evaluated. The equivalent for the data is a **datasheet** or dataset card<sup class="c"><a href="#r2">[2]</a></sup>, and in chemistry both overlap with the OECD/QMRF reporting of Unit 23 and with the DOME recommendations for machine learning in the life sciences<sup class="c"><a href="#r8">[8]</a><a href="#r9">[9]</a><a href="#r10">[10]</a></sup>.



In [ ]:
ev <- data.frame(obs = y[TE], pred = pred_te,
                 domain = ifelse(inside, "in domain", "outside"))
g1 <- ggplot(ev, aes(obs, pred, colour = domain)) +
  geom_point(size = 0.5, alpha = 0.5) + geom_abline(slope = 1, intercept = 0, linewidth = 0.3) +
  scale_colour_manual(values = c(`in domain` = OKABE[1], outside = OKABE[2]), name = NULL) +
  coord_equal(xlim = c(3.2, 10.3), ylim = c(3.2, 10.3)) +
  labs(x = "observed pActivity", y = "predicted", title = "Scaffold test set") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
cov2 <- data.frame(level = c("80%", "90%", "95%"), width = 2 * as.numeric(QS),
                   covered = coverage$empirical_all)
g2 <- ggplot(cov2, aes(level, width)) + geom_col(fill = OKABE[1], width = 0.6) +
  geom_text(aes(label = sprintf("width %.2f\ncovered %.0f%%", width, 100 * covered)),
            vjust = -0.2, size = 2.3) +
  ylim(0, max(cov2$width) * 1.35) +
  labs(x = "nominal confidence", y = "interval width (log units)", title = "Conformal intervals")
dd <- rbind(data.frame(d = d_train, what = "training"),
            data.frame(d = d_test, what = "scaffold test"),
            data.frame(d = groups[["screening decoys"]]$dist, what = "decoys"))
g3 <- ggplot(dd, aes(d, fill = what)) + geom_density(alpha = 0.45, colour = NA) +
  geom_vline(xintercept = AD_THRESHOLD, linetype = 2, linewidth = 0.4) +
  scale_fill_manual(values = c(training = OKABE[3], `scaffold test` = OKABE[1], decoys = OKABE[2]),
                    name = NULL) +
  labs(x = "distance to the training set", y = "density", title = "Applicability domain") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
print(g1 | g2 | g3)
cat(sprintf("in-domain test RMSE %.3f vs outside %.3f\n",
            sqrt(mean((pred_te[inside] - y[TE][inside])^2)),
            sqrt(mean((pred_te[!inside] - y[TE][!inside])^2))))
cat(sprintf("interval widths at 80/90/95%%: %s log units; empirical coverage %s\n",
            paste(round(2 * as.numeric(QS), 2), collapse = ", "),
            paste(sprintf("%.0f%%", 100 * coverage$empirical_all), collapse = ", ")))
cat(sprintf("median distance: training %.2f, test %.2f, decoys %.2f; threshold %.2f\n",
            median(d_train), median(d_test), median(groups[["screening decoys"]]$dist), AD_THRESHOLD))



<p class="caption"><b>Figure 6.</b> The three pieces of evidence behind the model card. Left: predicted against observed potency on the scaffold-split test set, coloured by the applicability-domain verdict. Middle: the width of the conformal interval at three confidence levels with the empirical coverage printed above each bar — the widths grow steeply while the coverage stays at or below nominal, which is what a violated exchangeability assumption looks like. Right: the distribution of distance to the training set for training compounds, for the scaffold-split test compounds and for decoys from an unrelated chemistry, with the threshold as a dashed line; the separation between the green and red distributions is why the flag is informative.</p>



In [ ]:
full <- predict_compound(ex$std_smiles[TE[1]], explain = TRUE)
att  <- do.call(rbind, lapply(full$top_features, as.data.frame))
print(ggplot(att, aes(reorder(feature, contribution), contribution, fill = contribution > 0)) +
  geom_col(width = 0.6, show.legend = FALSE) + coord_flip() +
  scale_fill_manual(values = c(`TRUE` = OKABE[3], `FALSE` = OKABE[2])) +
  labs(x = NULL, y = "change in the predicted pActivity when the bit is switched off",
       title = "Largest bit-ablation attributions for one compound"))
t_shap <- system.time({
  suppressPackageStartupMessages(library(iml))
  bg <- as.data.frame(X[sample(TR, 20), , drop = FALSE])
  pr <- Predictor$new(fit, data = bg, predict.function = function(model, newdata)
    predict(model, data = newdata)$predictions)
  sh <- Shapley$new(pr, x.interest = as.data.frame(X[TE[1], , drop = FALSE]), sample.size = 2)
})[["elapsed"]]
cat(sprintf("bit-ablation explanation: %.0f ms for this molecule (%d set bits)\n",
            1e3 * (t_one - t_noex), sum(X[TE[1], 1:2048] > 0)))
cat(sprintf("iml::Shapley with a 20-compound background and 2 Monte-Carlo samples: %.1f s\n", t_shap))
cat(sprintf("a usable setting (100 background compounds, 20 samples) costs about %.0f s per molecule\n",
            t_shap * (100 / 20) * (20 / 2)))
cat("which is why the deployed contract uses ablation and the model card says so\n")



<p class="caption"><b>Figure 7.</b> Per-prediction explanation for one compound: the five fingerprint bits whose removal changes the predicted potency most. The bits identify substructural environments and are not self-explanatory — the standard limitation of explaining a fingerprint model<sup class="c"><a href="#r13">[13]</a></sup> — which is why the interface shows the nearest measured compounds beside them. The timings printed beneath the figure are the reason this is an ablation rather than a Shapley value<sup class="c"><a href="#r11">[11]</a><a href="#r12">[12]</a></sup>: TreeSHAP has no R implementation for <code>ranger</code>, and the model-agnostic Monte-Carlo estimator in <code>iml</code> costs minutes per molecule over 2,059 features.</p>



In [ ]:
inside_rmse  <- sqrt(mean((pred_te[inside] - y[TE][inside])^2))
outside_rmse <- sqrt(mean((pred_te[!inside] - y[TE][!inside])^2))
CARD <- sprintf(
' # Model card - ROCK1 potency predictor (rock1-potency-rf), version %s

## Model details
- Random-forest regression (R package ranger %s, 300 trees, seed %d, all other hyperparameters at
  package defaults) on Morgan fingerprints (radius 2, 2,048 bits, taken from the frozen RDKit file
  shipped with the course) concatenated with %d physicochemical descriptors (%d features in total).
- Output: pActivity = -log10(IC50 or Ki in molar units) for human ROCK1 (ChEMBL CHEMBL3231,
  UniProt Q13464), with an inductive conformal interval and an applicability-domain verdict.
- Developed by the Laboratorio de Bioinformatica y Quimica Computacional (LBQC), Universidad
  Catolica del Maule, as teaching and internal prioritisation software. Code MIT; model and data
  CC BY-SA 3.0, inherited from ChEMBL.

## Intended use
- Intended: ranking candidate compounds for ROCK1 inhibition within the applicability domain;
  triage of purchasable libraries before docking or assay; teaching.
- Not intended: absolute potency prediction for decision-making without experimental confirmation;
  selectivity prediction against other kinases (never trained for it); any clinical, diagnostic or
  safety-related use; prediction for molecules outside the domain defined below.
- Users: medicinal chemists and computational colleagues; outputs are estimates, not measurements.

## Training data
- Source: ChEMBL release 37, target CHEMBL3231, IC50/Ki/Kd, human (sha256 %s).
- Curation: standardised parent structures keyed by InChIKey; median of exact measurements per
  compound; censored records excluded from the regression target (%d compounds). %d compounds with
  exact values remain.
- Split: deterministic Bemis-Murcko scaffold split - train %d, calibration %d, test %d; no scaffold
  is shared between the parts.
- Known biases: the data aggregate many laboratories and assay formats; actives are over-represented
  relative to any realistic library; the chemistry is concentrated in a few scaffold families.

## Metrics (scaffold hold-out, n = %d)
- RMSE %.2f, MAE %.2f, R2 %.2f log units.
- Pooled experimental standard deviation of replicate measurements in the source data: about 0.5 log
  units, so the model error is within a small factor of the measurement itself.
- Conformal half-widths %s log units at 80/90/95%% nominal confidence; empirical coverage on the
  scaffold test set %s - at or below nominal, because a scaffold split violates exchangeability.
- By domain: RMSE %.2f inside the applicability domain against %.2f outside.

## Applicability domain
- Definition: 1 minus the mean Tanimoto similarity (Morgan radius 2, 2,048 bits) to the five most
  similar training compounds; inside when that distance is at most %.2f, the 95th percentile of the
  training compounds own distances.
- %.0f%% of the scaffold-split test compounds fall inside. Predictions for molecules outside revert
  towards the training mean of %.2f and must be treated as refusals, not as estimates.
- Hard limits enforced before prediction: %d-%d heavy atoms, common organic elements only, largest
  fragment of a mixture, and a structure for which a fingerprint can be computed.

## Limitations and ethical considerations
- Public bioactivity data are heterogeneous and incomplete; absence of a measurement is not evidence
  of inactivity, and censored values were excluded rather than modelled.
- The model predicts potency only. It says nothing about selectivity, toxicity, solubility,
  synthesisability or suitability for any use in animals or humans.
- Explanations are single-bit ablations, not Shapley values; they ignore interactions between bits.
- Deployed as a web service it processes structures that may be confidential; the service stores no
  structures and logs only counts and timings.

## Maintenance
- Version %s, built from the frozen curated table; artefact sha256 %s.
- Retraining when ChEMBL adds a substantial number of ROCK1 measurements (reviewed yearly) or when a
  dependency upgrade changes the environment check.
- Contact: the laboratory. Issues and the deposited code: the repository linked from the service.',
  MODEL_VERSION, as.character(packageVersion("ranger")), SEED, length(DESC), ncol(X),
  manifest$training_data$sha256, sum(df$censored), nrow(ex), length(TR), length(CA), length(TE),
  length(TE), rmse_te, mae_te, r2_te,
  paste(sprintf("%.2f", as.numeric(QS)), collapse = ", "),
  paste(sprintf("%.0f%%", 100 * coverage$empirical_all), collapse = ", "),
  inside_rmse, outside_rmse, AD_THRESHOLD, 100 * mean(inside), mean(y[TR]),
  MIN_HEAVY, MAX_HEAVY, MODEL_VERSION, manifest$artefact$sha256)
writeLines(sub("^ ", "", CARD), file.path(TMP, "MODEL_CARD.md"))
cat(sub("^ ", "", CARD))
cat(sprintf("\n\n[%d words]\n", lengths(gregexpr("\\S+", CARD))))


In [ ]:
DATA_CARD <- sprintf(
' # Dataset card - ROCK1 curated bioactivity table

- **What it is.** %d compounds with measured affinity for human ROCK1, one row per standardised
  parent structure, derived from %d activity records.
- **Provenance.** ChEMBL 37 (CHEMBL3231); standardisation and aggregation by the curation package of
  Unit 24; sha256 %s.
- **Composition.** %d compounds with at least one exact measurement (the modelling set) and %d whose
  measurements are all censored (">" or "<"), kept and flagged. Potency range %.2f-%.2f, median %.2f.
- **Collection process.** Aggregated by ChEMBL from the medicinal-chemistry literature: many
  laboratories, assay formats and years. Replicate measurements of the same compound differ by about
  0.5 log units on average.
- **Known gaps and biases.** Publication bias towards potent compounds; scaffold concentration
  (%d scaffolds for %d compounds, many singletons); no negative data in the usual sense; no
  information about compound purity or aggregation.
- **Representation shipped with the course.** Morgan fingerprints computed once with RDKit and frozen
  as a sparse (id, bit) file, because the R edition has no chemistry toolkit; the SDF of the same
  structures is shipped so that they can be recomputed.
- **Recommended uses.** Model development with scaffold- or cluster-based splits; teaching.
  **Discouraged.** Treating censored records as exact; random splits for performance claims about new
  chemotypes; comparing values across assay types without care.
- **Licence.** CC BY-SA 3.0 (ChEMBL); cite Zdrazil et al. 2024, doi:10.1093/nar/gkad1004.',
  nrow(df), sum(df$n_measurements), manifest$training_data$sha256, nrow(ex), sum(df$censored),
  min(ex$pActivity), max(ex$pActivity), median(ex$pActivity), uniqueN(ex$scaffold_smiles), nrow(ex))
writeLines(sub("^ ", "", DATA_CARD), file.path(TMP, "DATA_CARD.md"))
cat(sub("^ ", "", DATA_CARD))



Both cards are generated from the variables the document computed, which is the only way they stay true after a retraining. The model card is the document the service links to from its footer and from `/model`; the dataset card travels with the deposited table.

## 10. Deploying it, keeping it alive, and switching it off

The deployment itself is the easy part. A `shiny` application goes to shinyapps.io, to a Posit Connect instance or to a Hugging Face Space with a Docker runtime, by pointing the platform at a repository containing `app.R`, a pinned `renv.lock` and the model artefact (or a download step for it); a `plumber` service goes into the same container, started by `plumber::pr_run()`. What follows deployment is the part that is routinely skipped.

**Secrets.** API keys, tokens and database URLs are injected as environment variables through the platform's secrets interface and read with `Sys.getenv()`. They are never in the repository, never in the image, never in a notebook and never printed in a log — a key committed once is a key to rotate, because the history keeps it.

**Logging.** Log counts, latencies, versions and error classes; do not log user structures. If usage analysis requires some record of what was asked, store a hash, state that in the privacy notice, and set a retention period.

**Versioning, twice over.** The application and the model have independent versions, and the response carries both: `model_version` in every payload, and the app version in the footer. A user who reports "the number changed" must be able to tell which of the two moved.

**Monitoring.** A `/health` endpoint the platform polls; an alert on error rate and on latency; and — specific to scientific services — a periodic check of the *distribution of what users submit*. A rising fraction of out-of-domain submissions is the signal that the tool is being used for something it was not built for, and that is a scientific problem before it is an engineering one<sup class="c"><a href="#r4">[4]</a><a href="#r5">[5]</a></sup>.

**Archiving.** The repository, the model artefact and the manifest are deposited and given a DOI at each release, which is what makes the software citable<sup class="c"><a href="#r27">[27]</a></sup> and what the FAIR principles for research software ask of a deposited tool<sup class="c"><a href="#r26">[26]</a></sup>.

**Maintenance and sunset.** A deployed tool is a commitment with no natural end, and free tiers sleep, change their runtimes and occasionally disappear. The group's policy, written into the model card and the README of every deployed tool, has four clauses: the service is maintained for a stated period after publication; dependencies are pinned and reviewed annually; if the tool is retired, the repository, the model artefact and an archived DOI remain, with instructions to run it locally; and the publication never cites the URL alone — it cites the archived artefact, because the paper must stay reproducible after the server is gone.



In [ ]:
life <- data.frame(
  month = c(0, 6, 12, 18, 30),
  label = c("model v1.0.0\nreleased with the paper", "dependency review\n(renv.lock pinned)",
            "ChEMBL update:\nretrain -> v1.1.0", "usage review:\nout-of-domain rising",
            "maintenance window ends;\narchived at a DOI"),
  colour = c(OKABE[1], OKABE[3], OKABE[1], OKABE[4], OKABE[2]))
life$side <- rep(c(1, -1), length.out = nrow(life))
print(ggplot(life, aes(month, 0)) +
  geom_hline(yintercept = 0, colour = "grey70", linewidth = 0.4) +
  geom_segment(aes(xend = month, y = 0, yend = 0.25 * side), colour = "grey70", linewidth = 0.3) +
  geom_point(colour = life$colour, size = 3.2) +
  geom_text(aes(y = 0.42 * side, label = label), size = 2.4, lineheight = 1.05) +
  scale_x_continuous(breaks = c(0, 6, 12, 18, 24, 30),
                     labels = c("release", "+6 m", "+12 m", "+18 m", "+24 m", "+30 m")) +
  coord_cartesian(ylim = c(-0.75, 0.75)) +
  labs(x = NULL, y = NULL, title = "A deployed model has a lifetime, and it must be written down") +
  theme(axis.text.y = element_blank(), panel.grid.major.y = element_blank()))
cat("the commitments in the model card are what make the last event an orderly retirement\n")
cat("rather than a dead link in a published paper\n")



<p class="caption"><b>Figure 8.</b> The life cycle the group commits to when it deploys a model: release with the paper, a dependency review every six months, retraining when the source database changes materially, a usage review that watches the fraction of out-of-domain submissions, and an announced end of the maintenance window with the artefact archived under a DOI. The dates are a policy, not a prediction; what matters is that they exist in writing before the tool is published.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Write the sunset paragraph for a tool your own project would deploy: how long it is maintained, who maintains it after the author graduates, what is archived and where, and what a user should do when the service stops. Then check the last three computational tools you used from a paper: how many are still running, and how many tell you what to do if they are not?</p></div>

## 11. Summary

**Shipping a model means shipping a contract**, not a file: a prediction, an interval with measured coverage, an applicability-domain verdict and evidence a chemist can judge. The same function serves the application and the API, which is what keeps the two consistent.

**Serialisation is not documentation.** `saveRDS()` stores an object with no record of the packages that produced it, and in R &mdash; a language whose design has always coupled data analysis to the graphics and environment that produced it<sup class="c"><a href="#r3">[3]</a></sup> &mdash; it also stores the *environment* of any function you save — the careless closure measured here carried ten megabytes of an unrelated matrix into the file alongside the model. A manifest with package versions, data checksums, metrics and licences turns a binary into an artefact, and a start-up environment check turns a silent incompatibility into a refusal<sup class="c"><a href="#r5">[5]</a></sup> &mdash; the same requirement that reproducibility standards for machine learning in the life sciences place on any published predictor<sup class="c"><a href="#r6">[6]</a></sup>.

**Validate before predicting.** Every rejection class — empty, too long, unparsable, unsupported elements, outside the size window, a mixture, and (in this R build) not featurisable — returns a reason rather than a number or a traceback, and the standardised structure is echoed back so the user sees what was actually scored.

**Silent extrapolation is the failure mode of deployment.** Asked about decoys and about the most dissimilar chemistry available, the model returned confident-looking potencies whose spread collapsed towards its training mean, while the distance measure flagged them and the prediction itself gave no sign. Within the scaffold test set the same measure barely graded the error at all: a domain flag is a filter on what may be asked, not a correction to the answer.

**Model cards and dataset cards are the instrument for that**<sup class="c"><a href="#r1">[1]</a><a href="#r2">[2]</a></sup>: intended and unintended uses, provenance and biases of the data, metrics under the honest split, the domain definition with its threshold, limitations, and a maintenance commitment — generated from the analysis, like everything else in Unit 23.

**Deployment creates obligations**: secrets from the environment, logs without chemistry, two version numbers, a health endpoint, a watch on the distribution of what users submit, and a written sunset policy with an archived artefact so that the paper outlives the server.

<div class="box bridge"><h4>Bridge</h4>
<p>The model served here was built in <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a>, validated on the splits of <a href="../U12_Validation/index.html">Unit 12</a>, explained with the attributions of <a href="../U16_Interpretability/index.html">Unit 16</a>, and given its applicability domain and conformal intervals in <a href="../U17_Uncertainty_AD/index.html">Unit 17</a>; its data and their biases come from <a href="../U6_Data_Quality/index.html">Unit 6</a>, its reporting from <a href="../U23_Communication/index.html">Unit 23</a>, the package and tests behind the service from <a href="../U24_Software_Engineering/index.html">Unit 24</a>, and the container it is deployed in from <a href="../U25_Scaling_HPC/index.html">Unit 25</a>. The capstone, <a href="../U27_Capstone/index.html">U27</a>, asks for a model card as part of the deliverable. In the sister courses, <a href="../../../Cheminformatics/U14_Validation/index.html">Validation</a> covers the QSAR reporting conventions a model card must satisfy, <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Descriptors and fingerprints</a> explains the representation whose bits appear in the attributions, and the R edition of the AI course, <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">Model evaluation</a>, covers the metrics quoted in the card. The Python edition of this unit, which deploys the same model with Streamlit and FastAPI, is <a href="../../Python/U26_Apps_Deployment/index.html">here</a>.</p></div>

## 12. Self-assessment

<details><summary>1. A collaborator asks for "the model". What do you send, and why is the file not enough?</summary>
<p>The model file, a manifest (package versions, data checksum, hyperparameters, metrics, split), the inference code including validation and the applicability-domain definition, and the model card. The `.rds` alone is an object whose behaviour depends on the version of <code>ranger</code> that loads it, and which says nothing about what it was trained on, how well it does on new chemistry, or where it must not be used.</p></details>

<details><summary>2. Why must the conformal quantile be computed on a calibration set rather than on the training residuals?</summary>
<p>Because a random forest fits its training data almost perfectly, so training residuals are far smaller than prediction errors on new compounds; intervals built from them would be far too narrow. The calibration set is data the model never saw during fitting, which is what makes the rank argument behind conformal validity apply. In the three-way split used here the calibration compounds also share no scaffold with the training compounds.</p></details>

<details><summary>3. Work out the conformal index for n = 199 calibration residuals at 95 % confidence, and say what happens at n = 15.</summary>
<p>k = ceiling(200 x 0.95) = 190, so the half-width is the 190th smallest of the 199 residuals. At n = 15, ceiling(16 x 0.95) = 16 > 15: the required index does not exist, so a 95 % interval cannot be formed. The rule is n >= 1/eps - 1, that is at least 19 calibration points for 95 %; with fewer, report a lower confidence and say so.</p></details>

<details><summary>4. The tool returns a plausible potency for a molecule from an unrelated chemistry. Is the model broken?</summary>
<p>No - it is doing what regression models do. A random forest averages the training leaves its input falls into, so an unfamiliar molecule receives something close to the training mean, with no indication that the question was inappropriate. The defect is in the interface if it shows that number without the applicability-domain verdict; in Section 6 every one of the most dissimilar decoys was flagged, at a mean domain distance more than half again the threshold, and that flag — not the number — is the answer the user needs.</p></details>

<details><summary>5. Give three input classes your validation must reject and the message for each.</summary>
<p>An unparsable string ("unbalanced parentheses: not a parsable SMILES string" - a compound name or a truncated paste is the usual cause); a molecule with unsupported elements ("unsupported elements: Pt" - the training set is small organic molecules); and a structure outside the size window ("about 120 heavy atoms, outside the supported range 3-70"). A mixture or salt is a fourth class, handled differently: the largest fragment is kept and the user is told, because silently predicting for a different species is worse than refusing.</p></details>

<details><summary>6. Where does a shiny application load its model, and why not inside the server function?</summary>
<p>At the top level of <code>app.R</code>, which runs once per R process and is shared by every session. Loading inside <code>server()</code> would reload the bundle for every user who connects, and loading inside a <code>reactive()</code> would reload it on every input change. The reactive graph is the other half of the answer: one <code>reactive()</code> holds the prediction and every output reads it, so the model is called once per input change regardless of how many outputs there are.</p></details>

<details><summary>7. What does FastAPI's schema give that the plumber service has to do by hand?</summary>
<p>Declarative validation before any user code runs, a machine-readable 422 naming the offending field, and an OpenAPI specification generated from the same declaration. <code>plumber</code> has no type annotations to read, so the batch cap, the allowed confidence levels and the type of the <code>smiles</code> field are checked in explicit code — which works, but means the documentation and the implementation can drift apart unless the checks and the annotations are reviewed together.</p></details>

<details><summary>8. Which four tests would you write for the API, and what does each protect?</summary>
<p>A malformed payload returns 422 (the validation is active); a request without the key returns 401 when a key is configured (the secret is enforced, not merely present); an unparsable structure returns a 200 with <code>status: "rejected"</code> and a reason (one bad molecule does not fail a batch); and a reference molecule returns the same prediction to three decimals (a golden test protecting against an unnoticed model or dependency change). The first three call the handler directly, as Section 8 does; the fourth belongs with the package tests of Unit 24.</p></details>

<details><summary>9. A user reports that the prediction for their compound changed between Monday and Friday. How do you diagnose it?</summary>
<p>Ask for the <code>model_version</code> from the response they kept, and compare with the current one from <code>/model</code>. If the versions differ, the model was retrained and the NEWS file says why. If they are the same, the environment changed under the service: compare the running package versions with the manifest — which is the check that should have refused to start — and look at the logs for a restart. This is why the version travels in every response.</p></details>

<details><summary>10. What belongs in the "not intended" section of a model card for this predictor?</summary>
<p>Absolute potency for decisions without experimental confirmation; selectivity against other kinases, which it was never trained on; anything clinical, diagnostic or safety-related; and prediction for molecules outside the stated applicability domain. The section exists because a model is used for whatever it plausibly answers, and the author is the only person who knows what it cannot do.</p></details>

<details><summary>11. Your free-tier application has been asleep for three months and the paper is published. What should already be in place?</summary>
<p>An archived release of the repository and the model artefact with a DOI, cited in the paper instead of (or alongside) the URL; a README with local installation instructions; a stated maintenance window; and a named contact. The service going to sleep is then an inconvenience rather than a reproducibility failure — which is the practical content of a sunset policy.</p></details>

<details><summary>12. A colleague wants to score a 200,000-compound catalogue through your API. What do you tell them?</summary>
<p>Not through the API: at the per-compound latency measured in Section 5 the arithmetic alone is of the order of ten minutes of uninterrupted prediction, and as 200,000 separate HTTP calls it is far more; the batch endpoint is capped at 200 structures for exactly that reason. Give them the package and the model artefact so they can run it locally or on the cluster with the parallelisation of Unit 25, and point out that most of those compounds will be outside the applicability domain, so the useful first step is to filter by distance to the training set before predicting anything.</p></details>

## Glossary

<dl class="gloss">
<dt>Applicability domain</dt><dd>The region of chemical space where a model's predictions are supported by training data; here, 1 minus the mean Tanimoto similarity to the five nearest training compounds, with a threshold from the training distribution.</dd>
<dt>Bit ablation</dt><dd>A local explanation for a fingerprint model: the change in the prediction when one set bit is switched off. Cheap, counterfactual, and blind to interactions between bits.</dd>
<dt>Conformal prediction (inductive)</dt><dd>A distribution-free method giving intervals with a stated coverage, built from the quantile of residuals on a held-out calibration set.</dd>
<dt>Dataset card / datasheet</dt><dd>A structured description of a data set: provenance, composition, collection process, biases, recommended and discouraged uses, licence.</dd>
<dt>Environment of a closure</dt><dd>The frame in which a function was defined, saved with it by <code>saveRDS</code>; the reason a one-line function can produce a multi-megabyte file.</dd>
<dt>Health endpoint</dt><dd>A trivial route (<code>/health</code>) that a hosting platform or monitor polls to decide whether a service is alive.</dd>
<dt>Model card</dt><dd>A short structured document stating a model's intended and unintended uses, training data, metrics under a named evaluation, applicability domain, limitations and maintenance.</dd>
<dt>plumber</dt><dd>The R package that turns annotated functions into HTTP endpoints; validation of the request body is written by hand.</dd>
<dt>Reactive (shiny)</dt><dd>An expression that recomputes only when its inputs change; the unit of caching in a shiny application.</dd>
<dt>saveRDS / readRDS</dt><dd>R's single-object serialisation; records no package versions, and stores the environment of any function it is given.</dd>
<dt>Silent extrapolation</dt><dd>A confident-looking prediction for an input outside the model's domain; detectable only by a domain check, never by the prediction itself.</dd>
<dt>Sunset policy</dt><dd>The written commitment describing how long a deployed tool is maintained, what is archived, and what users should do when it stops.</dd>
<dt>testServer</dt><dd><code>shiny::testServer()</code>: drives a server function's reactive graph in-process, so an application's logic can be tested without a browser or a port.</dd>
</dl>



In [ ]:
produced <- sort(basename(list.files(TMP, pattern = "rock1_rf|manifest|CARD|closure")))
cat("artefacts written by this document:", paste(produced, collapse = ", "), "\n")
invisible(file.remove(list.files(TMP, pattern = "rock1_rf|manifest|CARD|closure", full.names = TRUE)))
cat("artefacts removed:", length(list.files(TMP, pattern = "rock1_rf|manifest|CARD|closure")) == 0, "\n")



In a real deployment those files are the release: the `.rds`, the manifest, the two cards and the application source, tagged in the repository and archived under a DOI. Here they were written into the session's temporary directory and deleted, because a document that leaves artefacts behind is a document that will one day be run twice.

## References and further reading

<ol class="refs">
<li id="r1">Mitchell M, Wu S, Zaldivar A, et al. Model cards for model reporting. In: <em>Proc. Conf. on Fairness, Accountability, and Transparency (FAT*)</em>; 2019:220-229. <a href="https://doi.org/10.1145/3287560.3287596">doi:10.1145/3287560.3287596</a></li>
<li id="r2">Gebru T, Morgenstern J, Vecchione B, et al. Datasheets for datasets. <em>Commun ACM</em>. 2021;64(12):86-92. <a href="https://doi.org/10.1145/3458723">doi:10.1145/3458723</a></li>
<li id="r3">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r4">Paleyes A, Urma RG, Lawrence ND. Challenges in deploying machine learning: a survey of case studies. <em>ACM Comput Surv</em>. 2022;55(6):114. <a href="https://doi.org/10.1145/3533378">doi:10.1145/3533378</a></li>
<li id="r5">Haibe-Kains B, Adam GA, Hosny A, et al. Transparency and reproducibility in artificial intelligence. <em>Nature</em>. 2020;586:E14-E16. <a href="https://doi.org/10.1038/s41586-020-2766-y">doi:10.1038/s41586-020-2766-y</a></li>
<li id="r6">Heil BJ, Hoffman MM, Markowetz F, Lee SI, Greene CS, Hicks SC. Reproducibility standards for machine learning in the life sciences. <em>Nat Methods</em>. 2021;18:1132-1135. <a href="https://doi.org/10.1038/s41592-021-01256-7">doi:10.1038/s41592-021-01256-7</a></li>
<li id="r7">Artrith N, Butler KT, Coudert FX, et al. Best practices in machine learning for chemistry. <em>Nat Chem</em>. 2021;13:505-508. <a href="https://doi.org/10.1038/s41557-021-00716-z">doi:10.1038/s41557-021-00716-z</a></li>
<li id="r8">Walsh I, Fishman D, Garcia-Gasulla D, et al. DOME: recommendations for supervised machine learning validation in biology. <em>Nat Methods</em>. 2021;18:1122-1127. <a href="https://doi.org/10.1038/s41592-021-01205-4">doi:10.1038/s41592-021-01205-4</a></li>
<li id="r9">Bender A, Schneider N, Segler M, Walters WP, Engkvist O, Rodrigues T. Evaluation guidelines for machine learning tools in the chemical sciences. <em>Nat Rev Chem</em>. 2022;6:428-442. <a href="https://doi.org/10.1038/s41570-022-00391-9">doi:10.1038/s41570-022-00391-9</a></li>
<li id="r10">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r11">Lundberg SM, Lee SI. A unified approach to interpreting model predictions. In: <em>NeurIPS 30</em>; 2017. <a href="https://doi.org/10.48550/arXiv.1705.07874">doi:10.48550/arXiv.1705.07874</a></li>
<li id="r12">Lundberg SM, Erion G, Chen H, et al. From local explanations to global understanding with explainable AI for trees. <em>Nat Mach Intell</em>. 2020;2:56-67. <a href="https://doi.org/10.1038/s42256-019-0138-9">doi:10.1038/s42256-019-0138-9</a></li>
<li id="r13">Jimenez-Luna J, Grisoni F, Schneider G. Drug discovery with explainable artificial intelligence. <em>Nat Mach Intell</em>. 2020;2:573-584. <a href="https://doi.org/10.1038/s42256-020-00236-4">doi:10.1038/s42256-020-00236-4</a></li>
<li id="r14">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r15">Norinder U, Carlsson L, Boyer S, Eklund M. Introducing conformal prediction in predictive modeling: a transparent and flexible alternative to applicability domain determination. <em>J Chem Inf Model</em>. 2014;54(6):1596-1603. <a href="https://doi.org/10.1021/ci5001168">doi:10.1021/ci5001168</a></li>
<li id="r16">Svensson F, Aniceto N, Norinder U, et al. Conformal regression for quantitative structure-activity relationship modeling - quantifying prediction uncertainty. <em>J Chem Inf Model</em>. 2018;58(5):1132-1140. <a href="https://doi.org/10.1021/acs.jcim.8b00054">doi:10.1021/acs.jcim.8b00054</a></li>
<li id="r17">Papadopoulos H, Proedrou K, Vovk V, Gammerman A. Inductive confidence machines for regression. In: <em>ECML 2002</em>. LNCS 2430. Springer; 2002:345-356. <a href="https://doi.org/10.1007/3-540-36755-1_29">doi:10.1007/3-540-36755-1_29</a></li>
<li id="r18">Vovk V, Gammerman A, Shafer G. <em>Algorithmic Learning in a Random World</em>. Springer; 2005. <a href="https://doi.org/10.1007/b106715">doi:10.1007/b106715</a></li>
<li id="r19">Sahigara F, Mansouri K, Ballabio D, Mauri A, Consonni V, Todeschini R. Comparison of different approaches to define the applicability domain of QSAR models. <em>Molecules</em>. 2012;17(5):4791-4810. <a href="https://doi.org/10.3390/molecules17054791">doi:10.3390/molecules17054791</a></li>
<li id="r20">Mervin LH, Johansson S, Semenova E, Giblin KA, Engkvist O. Uncertainty quantification in drug design. <em>Drug Discov Today</em>. 2021;26(2):474-489. <a href="https://doi.org/10.1016/j.drudis.2020.11.027">doi:10.1016/j.drudis.2020.11.027</a></li>
<li id="r21">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <em>J Chem Inf Model</em>. 2018;58(5):916-932. <a href="https://doi.org/10.1021/acs.jcim.7b00403">doi:10.1021/acs.jcim.7b00403</a></li>
<li id="r22">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r23">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r24">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r25">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r26">Chue Hong NP, Katz DS, Barker M, et al. Introducing the FAIR principles for research software. <em>Sci Data</em>. 2022;9:622. <a href="https://doi.org/10.1038/s41597-022-01710-x">doi:10.1038/s41597-022-01710-x</a></li>
<li id="r27">Smith AM, Katz DS, Niemeyer KE. Software citation principles. <em>PeerJ Comput Sci</em>. 2016;2:e86. <a href="https://doi.org/10.7717/peerj-cs.86">doi:10.7717/peerj-cs.86</a></li>
<li id="r28">Stokes JM, Yang K, Swanson K, et al. A deep learning approach to antibiotic discovery. <em>Cell</em>. 2020;180(4):688-702. <a href="https://doi.org/10.1016/j.cell.2020.01.021">doi:10.1016/j.cell.2020.01.021</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H. <em>Mastering Shiny</em>. O'Reilly; 2021 (free online). - The reactive model, modules and testing, including <code>testServer</code>.</li>
<li>Huyen C. <em>Designing Machine Learning Systems</em>. O'Reilly; 2022. - Deployment, monitoring and the failure modes of models in production.</li>
<li>Molnar C. <em>Interpretable Machine Learning</em>. 2nd ed.; 2022 (free online). - The attribution methods behind the explanation panel, and what an ablation is not.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>Shiny documentation and the testing article: <a href="https://shiny.posit.co/">shiny.posit.co</a>.</li>
<li><code>plumber</code> documentation, including filters, serialisers and OpenAPI: <a href="https://www.rplumber.io/">rplumber.io</a>.</li>
<li><code>vetiver</code>, which packages a fitted model with its metadata and generates the serving code: <a href="https://vetiver.posit.co/">vetiver.posit.co</a>.</li>
<li>Hugging Face Spaces and the Model Card guidebook: <a href="https://huggingface.co/docs/hub/spaces">huggingface.co/docs/hub/spaces</a>, <a href="https://huggingface.co/docs/hub/model-cards">huggingface.co/docs/hub/model-cards</a>.</li>
<li>R serialization notes in <em>Writing R Extensions</em>, on what <code>saveRDS</code> does and does not record: <a href="https://cran.r-project.org/doc/manuals/r-release/R-exts.html">cran.r-project.org — R-exts</a>.</li>
</ul>
